In [ ]:
# inspect one file to make sure it has what we need
import xarray as xr
from pathlib import Path

directory = Path(
    "/global/cfs/projectdirs/m2637/ALCC/"
    "ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001"
)

files = list(directory.glob("*.nc"))

# print(f"Found {len(files)} NetCDF files")
# print(files[:5])

In [ ]:
file = files[0]

# print("Opening:")
# print(file)

ds = xr.open_dataset(file)

ds

In [ ]:
# open one mode and run and try combining for wind shear variables
from pathlib import Path
import xarray as xr

# source folder
folder = Path(
    "/global/cfs/projectdirs/m2637/ALCC/"
    "ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001"
)

variables = ["U200", "U850", "V200", "V850", "lat", "lon"]

# find all NetCDF files
files = sorted(folder.glob("*.nc"))

# print(f"Found {len(files)} files")

# combine files, retaining only the four variables
ds = xr.open_mfdataset(
    files,
    combine="by_coords",
    preprocess=lambda d: d[variables],
    parallel=False,
)

In [ ]:
# # check lat/lon dimensions

# print("Latitude dimensions:", ds["lat"].dims)
# print("Longitude dimensions:", ds["lon"].dims)

# print(
#     "Latitude range:",
#     float(ds["lat"].min()),
#     float(ds["lat"].max())
# )

# print(
#     "Longitude range:",
#     float(ds["lon"].min()),
#     float(ds["lon"].max())
# )

# print("Latitude units:", ds["lat"].attrs)
# print("Longitude units:", ds["lon"].attrs)

In [ ]:
# clip to NAtl region
import numpy as np
import xarray as xr

# coordinates at the first time step
lat = ds["lat"].isel(time=0)
lon = ds["lon"].isel(time=0)

# NAtl region: 120W–25E, 0N–90N
region_mask = (
    (lat >= 0) &
    (lat <= 90) &
    ((lon >= 240) | (lon <= 25))
)

# indices of grid cells inside the region
region_indices = np.flatnonzero(region_mask.values)

print(f"Selected {len(region_indices):,} grid cells")
print(f"Out of {ds.sizes['ncol']:,} total grid cells")

In [ ]:
# now clip to hurricane season
seasonal = ds.where(
    ds.time.dt.month.isin([6, 7, 8, 9, 10]),
    drop=True,
)

# select the North Atlantic grid cells
regional = seasonal.isel(ncol=region_indices)

print(regional)


In [ ]:
# # check whether the lat/lon coords vary over time
# lat_changes = bool(
#     (ds["lat"].isel(time=1) != ds["lat"].isel(time=0))
#     .any().compute()
# )

# lon_changes = bool(
#     (ds["lon"].isel(time=1) != ds["lon"].isel(time=0))
#     .any().compute()
# )

# print("Latitude changes with time:", lat_changes)
# print("Longitude changes with time:", lon_changes)

In [ ]:
# create 5deg lat/lon bins
variables = ["U200", "U850", "V200", "V850"]

# Coordinates are fixed, so use the first time step
lat = ds["lat"].isel(time=0).values
lon = ds["lon"].isel(time=0).values

# Convert longitude from 0–360 to -180–180
lon = ((lon + 180) % 360) - 180

# Use the same regional grid cells selected earlier
lat_reg = lat[region_indices]
lon_reg = lon[region_indices]

# Assign cells to 5-degree bins
lat_bin = np.floor(lat_reg / 5) * 5
lon_bin = np.floor(lon_reg / 5) * 5

# Build a working dataset without repeated lat/lon arrays
data = regional[variables].assign_coords(
    lat_bin=("ncol", lat_bin),
    lon_bin=("ncol", lon_bin),
)

print(data)

In [ ]:
# check 1: why my kernel is crashing; memory issue??
import psutil

# Overall system memory
mem = psutil.virtual_memory()
print(f"Total RAM: {mem.total / 1024**3:.2f} GB")
print(f"Available RAM: {mem.available / 1024**3:.2f} GB")
print(f"RAM used: {mem.percent}%")

# Size of your xarray object
print(data)

if hasattr(data, "nbytes"):
    print(f"Data size: {data.nbytes / 1024**3:.2f} GB")

In [ ]:
# check 2
import numpy as np

print("Chunks:", data["U200"].chunks)
print("Number of time steps:", data.sizes["time"])
print("Number of grid cells:", data.sizes["ncol"])

print(
    "Number of latitude bins:",
    np.unique(data["lat_bin"].values).size
)

print(
    "Number of longitude bins:",
    np.unique(data["lon_bin"].values).size
)

In [ ]:
# check 3
print(data)
print("\nDataset size in memory:")
print(f"{data.nbytes / 1024**3:.2f} GiB")

In [ ]:
# check 4
import dask
from dask.distributed import get_client

print("Dask version:", dask.__version__)
print("Configured scheduler:", dask.config.get("scheduler", default="not explicitly configured"))

try:
    client = get_client()
    print("\nActive Dask client:")
    print(client)
    print("\nWorker information:")
    for worker, info in client.scheduler_info()["workers"].items():
        print(
            worker,
            "threads:", info.get("nthreads"),
            "memory limit:", info.get("memory_limit", "unknown")
        )
except ValueError:
    print("\nNo active distributed Dask client.")
    print("Dask may be using its local scheduler.")

In [ ]:
# average per 5deg bin
# group grid cells by their 5-degree latitude/longitude bins
grouped = data.groupby(["lat_bin", "lon_bin"])

# average across grid cells in each bin, then across all selected times
binned = grouped.mean(dim="ncol").mean(dim="time")

print(binned)

In [ ]:
u = data["U200"]

# Sum over time and grid cells for each lat/lon bin
u_sum = u.groupby(["lat_bin", "lon_bin"]).sum(
    dim=("time", "ncol"), skipna=True
)

# Count valid values for each bin
u_count = u.groupby(["lat_bin", "lon_bin"]).count(
    dim=("time", "ncol")
)

# Final mean
mean_U200 = u_sum / u_count

print(mean_U200)